#  Model Training

## Purpose

This notebook trains and compares the five predictive models specified in the thesis for predicting firm export participation:

1. Logistic Regression (LOGIT)
2. LASSO Logistic Regression (LOGIT-LASSO)
3. Classification and Regression Tree (CART)
4. Random Forest
5. Bayesian Additive Regression Trees with Missingness Incorporated in Attributes (BART-MIA)

Model development is conducted separately for Kenya, Tanzania, and Uganda.

The training data generated in `03_model_preparation.ipynb` are used for model development. 
The held-out test observations remain untouched and are reserved for `05_model_evaluation.ipynb`.

Five-fold stratified cross-validation is used for the four scikit-learn models. 
Model selection is based on cross-validated ROC-AUC rather than performance on the held-out test set.

BART-MIA is trained separately because it uses a native Missingness Incorporated in Attributes
(MIA) mechanism and therefore must retain the relevant missing predictor values rather than
replacing them through conventional imputation.

## Countries

- Kenya
- Tanzania
- Uganda

## Primary model-selection criterion

ROC-AUC from cross-validation.

## Reproducibility

A fixed random seed is used throughout the model-development process.





## 1. Import Libraries

The following libraries are used for data handling, model estimation, cross-validation, hyperparameter tuning, performance measurement, and model persistence.

In [46]:
# IMPORT LIBRARIES
import os
import json
import pickle
import warnings
import subprocess
from pathlib import Path
import shutil




import textwrap
import numpy as np
import pandas as pd

# Scikit-learn
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")


RANDOM_STATE = 42
N_SPLITS = 5

np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")
print("Random state:", RANDOM_STATE)
print("Cross-validation folds:", N_SPLITS)

Libraries imported successfully.
Random state: 42
Cross-validation folds: 5


## 2. Paths and Model Configuration

The prepared modelling datasets are loaded from the `Data/model` directory.

The model-training outputs are stored separately from the raw and prepared datasets.

The target variable is `exporter`.

The 12 explanatory variables are defined explicitly to ensure that the modelling specification
remains consistent across all three countries and all five models.

In [21]:
# PATHS AND CONFIGURATION

BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "Data"
MODEL_DATA_DIR = DATA_DIR / "model"

RESULTS_DIR = BASE_DIR / "results"
TRAINING_DIR = RESULTS_DIR / "training"
MODEL_OUTPUT_DIR = RESULTS_DIR / "models"

TRAINING_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# COUNTRIES


countries = [
    "Kenya",
    "Tanzania",
    "Uganda"
]


# TARGET


target = "exporter"


# PREDICTORS


predictors = [
    "l1",
    "d2",
    "b7",
    "b2b",
    "h1",
    "c22b",
    "c36",
    "e6",
    "b3a",
    "l10",
    "c39",
    "k30"
]


# CROSS-VALIDATION


cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


print("Countries:", countries)
print("Target:", target)
print("Number of predictors:", len(predictors))
print("Model data directory:", MODEL_DATA_DIR)
print("Training output directory:", TRAINING_DIR)
print("Model output directory:", MODEL_OUTPUT_DIR)

Countries: ['Kenya', 'Tanzania', 'Uganda']
Target: exporter
Number of predictors: 12
Model data directory: ../Data/model
Training output directory: ../results/training
Model output directory: ../results/models


## 3. Load Prepared Modelling Data

The training and held-out test datasets created in `03_model_preparation.ipynb` are loaded.

The following files are used:

- `X_train`
- `y_train`
- `X_test`
- `y_test`

The test data are loaded only so that their structure can be verified. They are not used for
model selection or hyperparameter tuning in this notebook.

A separate modelling dataset containing the original training observations and their target
variable is also retained for the BART-MIA branch, because BART-MIA requires access to
missing predictor values.

In [22]:
# LOAD PREPARED DATA

X_train_data = {}
y_train_data = {}

X_test_data = {}
y_test_data = {}

for country in countries:

    X_train_path = MODEL_DATA_DIR / f"{country}_X_train.csv"
    y_train_path = MODEL_DATA_DIR / f"{country}_y_train.csv"

    X_test_path = MODEL_DATA_DIR / f"{country}_X_test.csv"
    y_test_path = MODEL_DATA_DIR / f"{country}_y_test.csv"

    X_train_data[country] = pd.read_csv(X_train_path)
    y_train_data[country] = pd.read_csv(y_train_path)

    X_test_data[country] = pd.read_csv(X_test_path)
    y_test_data[country] = pd.read_csv(y_test_path)

print("Prepared datasets loaded successfully.\n")

for country in countries:

    print(" " * 70)
    print(country)
    print(" " * 70)

    print("X_train:", X_train_data[country].shape)
    print("y_train:", y_train_data[country].shape)
    print("X_test :", X_test_data[country].shape)
    print("y_test :", y_test_data[country].shape)

Prepared datasets loaded successfully.

                                                                      
Kenya
                                                                      
X_train: (819, 12)
y_train: (819, 1)
X_test : (205, 12)
y_test : (205, 1)
                                                                      
Tanzania
                                                                      
X_train: (472, 12)
y_train: (472, 1)
X_test : (118, 12)
y_test : (118, 1)
                                                                      
Uganda
                                                                      
X_train: (484, 12)
y_train: (484, 1)
X_test : (121, 12)
y_test : (121, 1)


## 4. Separate Predictors and Target

The target variable is stored separately from the predictor matrix.

The `Unnamed: 0` column is an exported dataframe index and is not a substantive explanatory
variable. It is therefore removed before modelling.

The target variable is converted to a binary numerical representation:

- `0` = non-exporter
- `1` = exporter

The resulting objects are stored by country for subsequent model training.

In [23]:
# SEPARATE X AND y

X_train = {}
y_train = {}

X_test = {}
y_test = {}

for country in countries:

  
    # X


    Xtr = X_train_data[country].copy()
    Xte = X_test_data[country].copy()

    # Remove exported dataframe index if present
    Xtr = Xtr.drop(columns=["Unnamed: 0"], errors="ignore")
    Xte = Xte.drop(columns=["Unnamed: 0"], errors="ignore")

    # Retain only specified predictors
    Xtr = Xtr[predictors]
    Xte = Xte[predictors]

    
    # y
    

    ytr = y_train_data[country].copy()
    yte = y_test_data[country].copy()

    # Target is stored alongside an exported index
    if target not in ytr.columns:
        raise ValueError(
            f"{country}: target '{target}' not found in y_train."
        )

    if target not in yte.columns:
        raise ValueError(
            f"{country}: target '{target}' not found in y_test."
        )

    ytr = ytr[target]
    yte = yte[target]

    # Ensure numeric binary target
    ytr = pd.to_numeric(ytr, errors="coerce")
    yte = pd.to_numeric(yte, errors="coerce")

    X_train[country] = Xtr
    y_train[country] = ytr

    X_test[country] = Xte
    y_test[country] = yte

print("X/y separation completed.")

for country in countries:

    print("\n" + " " * 70)
    print(country)
    print(" " * 70)

    print("X_train shape:", X_train[country].shape)
    print("y_train shape:", y_train[country].shape)
    print("X_test shape :", X_test[country].shape)
    print("y_test shape :", y_test[country].shape)

    print("Training target:")
    print(y_train[country].value_counts(dropna=False).sort_index())

X/y separation completed.

                                                                      
Kenya
                                                                      
X_train shape: (819, 12)
y_train shape: (819,)
X_test shape : (205, 12)
y_test shape : (205,)
Training target:
exporter
0    676
1    143
Name: count, dtype: int64

                                                                      
Tanzania
                                                                      
X_train shape: (472, 12)
y_train shape: (472,)
X_test shape : (118, 12)
y_test shape : (118,)
Training target:
exporter
0    403
1     69
Name: count, dtype: int64

                                                                      
Uganda
                                                                      
X_train shape: (484, 12)
y_train shape: (484,)
X_test shape : (121, 12)
y_test shape : (121,)
Training target:
exporter
0    431
1     53
Name: count, dtype: int64


## 5. Training Data Structure Validation

Before model estimation, the prepared datasets are checked for consistency.

The validation confirms that:

- the expected 12 predictors are present;
- the target variable is available;
- training and target observations have matching row counts;
- the target contains only binary values;
- both classes are represented in each training sample;
- the held-out test data contain the same predictor structure.

The test data are inspected only for structural integrity and are not used in model fitting.

In [25]:
# STRUCTURE VALIDATION

for country in countries:

    print("\n" + " " * 70)
    print(country)
    print(" " * 70)

    Xtr = X_train[country]
    ytr = y_train[country]

    Xte = X_test[country]
    yte = y_test[country]

   
    # Predictor checks
    

    assert list(Xtr.columns) == predictors, (
        f"{country}: training predictors do not match specification."
    )

    assert list(Xte.columns) == predictors, (
        f"{country}: testing predictors do not match specification."
    )

    
    # Row checks
    

    assert len(Xtr) == len(ytr), (
        f"{country}: X_train and y_train row counts differ."
    )

    assert len(Xte) == len(yte), (
        f"{country}: X_test and y_test row counts differ."
    )

    assert len(Xtr) > 0
    assert len(Xte) > 0

   
    # Target checks
    

    train_values = set(ytr.dropna().unique())
    test_values = set(yte.dropna().unique())

    assert train_values.issubset({0, 1}), (
        f"{country}: invalid training target values."
    )

    assert test_values.issubset({0, 1}), (
        f"{country}: invalid testing target values."
    )

    assert ytr.notna().all(), (
        f"{country}: missing training target values detected."
    )

    # Both classes must exist for stratified CV
    assert ytr.nunique() == 2, (
        f"{country}: training target does not contain both classes."
    )

    print("Training observations:", len(Xtr))
    print("Testing observations :", len(Xte))
    print("Predictors           :", Xtr.shape[1])
    print("Training missing     :", Xtr.isna().sum().sum())
    print("Testing missing      :", Xte.isna().sum().sum())
    print("Training target      :", ytr.value_counts().to_dict())
    print("Structure validation : PASSED")


                                                                      
Kenya
                                                                      
Training observations: 819
Testing observations : 205
Predictors           : 12
Training missing     : 0
Testing missing      : 0
Training target      : {0: 676, 1: 143}
Structure validation : PASSED

                                                                      
Tanzania
                                                                      
Training observations: 472
Testing observations : 118
Predictors           : 12
Training missing     : 0
Testing missing      : 0
Training target      : {0: 403, 1: 69}
Structure validation : PASSED

                                                                      
Uganda
                                                                      
Training observations: 484
Testing observations : 121
Predictors           : 12
Training missing     : 0
Testing missing      : 0
Training target     

## 6. Define Cross-Validation Strategy

Five-fold stratified cross-validation is used for model selection.

Stratification preserves the class distribution across the folds.

ROC-AUC is used as the primary model-selection metric because the research
problem is a binary classification task with an imbalanced outcome.A fixed random seed ensures reproducibility.

In [ ]:

# CROSS-VALIDATION CONFIGURATION


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Cross-validation configuration")
print(" " * 50)
print("Method          : Stratified K-Fold")
print("Number of folds : 5")
print("Shuffle         : True")
print("Random state    :", RANDOM_STATE)
print("Primary metric  : ROC-AUC")
print("Test data       : Excluded")

Cross-validation configuration
                                                  
Method          : Stratified K-Fold
Number of folds : 5
Shuffle         : True
Random state    : 42
Primary metric  : ROC-AUC
Test data       : Excluded


# 7. model 1 - Logistic Regression (LOGIT)

Logistic Regression provides the baseline parametric classification model.

The model estimates the probability that a firm is an exporter as a function of the
specified explanatory variables.

Missing predictor values are handled within the modelling pipeline using median
imputation for numerical variables. Imputation is performed separately inside each
cross-validation training fold to avoid information leakage.

Predictors are standardized before estimation.

The regularization parameter `C` is tuned using five-fold stratified cross-validation.

The model is selected according to mean cross-validated ROC-AUC.

In [27]:

# LOGISTIC REGRESSION


logit_results = []
logit_models = {}

for country in countries:

    print("\n" + " " * 70)
    print(f"LOGISTIC REGRESSION — {country}")
    print(" " * 70)

    Xtr = X_train[country]
    ytr = y_train[country]

    # Pipeline:
    # 1. Median imputation
    # 2. Standardization
    # 3. Logistic regression
    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=RANDOM_STATE
            )
        )
    ])

    param_grid = {
        "model__C": [
            0.001,
            0.01,
            0.1,
            1.0,
            10.0,
            100.0
        ]
    }

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(Xtr, ytr)

    logit_models[country] = search.best_estimator_

    result = {
        "Country": country,
        "Model": "Logistic Regression",
        "Best_CV_ROC_AUC": search.best_score_,
        "Best_Parameters": search.best_params_
    }

    logit_results.append(result)

    print("Best parameters:")
    print(search.best_params_)

    print(
        f"Best mean CV ROC-AUC: "
        f"{search.best_score_:.4f}"
    )


                                                                      
LOGISTIC REGRESSION — Kenya
                                                                      
Best parameters:
{'model__C': 100.0}
Best mean CV ROC-AUC: 0.7952

                                                                      
LOGISTIC REGRESSION — Tanzania
                                                                      
Best parameters:
{'model__C': 0.001}
Best mean CV ROC-AUC: 0.7105

                                                                      
LOGISTIC REGRESSION — Uganda
                                                                      
Best parameters:
{'model__C': 0.01}
Best mean CV ROC-AUC: 0.8782


# 8. Model 2 - LASSO Logistic Regression (LOGIT-LASSO)

LASSO Logistic Regression extends the logistic regression specification by applying
an L1 penalty.

The L1 penalty can shrink coefficients toward zero and therefore provides a form of
regularization and variable selection.

The regularization strength is selected through five-fold stratified cross-validation.

As with the standard logistic regression model, missing values are handled within
the cross-validation pipeline and predictors are standardized before estimation.

ROC-AUC is used as the tuning criterion.

In [28]:
# LASSO LOGISTIC REGRESSION

lasso_results = []
lasso_models = {}

for country in countries:

    print("\n" + " " * 70)
    print(f"LASSO LOGISTIC REGRESSION — {country}")
    print(" " * 70)

    Xtr = X_train[country]
    ytr = y_train[country]

    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                max_iter=5000,
                random_state=RANDOM_STATE
            )
        )
    ])

    param_grid = {
        "model__C": [
            0.001,
            0.01,
            0.1,
            1.0,
            10.0,
            100.0
        ]
    }

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(Xtr, ytr)

    lasso_models[country] = search.best_estimator_

    result = {
        "Country": country,
        "Model": "LASSO Logistic Regression",
        "Best_CV_ROC_AUC": search.best_score_,
        "Best_Parameters": search.best_params_
    }

    lasso_results.append(result)

    print("Best parameters:")
    print(search.best_params_)

    print(
        f"Best mean CV ROC-AUC: "
        f"{search.best_score_:.4f}"
    )


                                                                      
LASSO LOGISTIC REGRESSION — Kenya
                                                                      
Best parameters:
{'model__C': 1.0}
Best mean CV ROC-AUC: 0.7956

                                                                      
LASSO LOGISTIC REGRESSION — Tanzania
                                                                      
Best parameters:
{'model__C': 100.0}
Best mean CV ROC-AUC: 0.6793

                                                                      
LASSO LOGISTIC REGRESSION — Uganda
                                                                      
Best parameters:
{'model__C': 0.1}
Best mean CV ROC-AUC: 0.8756


# 9. Model 3 - Classification and Regression Tree (CART)

A Classification and Regression Tree is estimated to provide a non-parametric,
rule-based alternative to the logistic specifications.

The tree recursively partitions firms according to predictor values.

Missing predictor values are handled within the modelling pipeline using median
imputation.

Tree complexity is controlled through hyperparameters including maximum depth,
minimum leaf size, and cost-complexity pruning.

The optimal specification is selected using five-fold stratified cross-validation
with ROC-AUC as the selection criterion.

In [29]:
# CART

cart_results = []
cart_models = {}

for country in countries:

    print("\n" + " " * 70)
    print(f"CART — {country}")
    print(" " * 70)

    Xtr = X_train[country]
    ytr = y_train[country]

    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            DecisionTreeClassifier(
                random_state=RANDOM_STATE
            )
        )
    ])

    param_grid = {
        "model__max_depth": [
            None,
            3,
            5,
            7,
            10
        ],
        "model__min_samples_leaf": [
            1,
            5,
            10,
            20
        ],
        "model__ccp_alpha": [
            0.0,
            0.001,
            0.005,
            0.01
        ]
    }

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(Xtr, ytr)

    cart_models[country] = search.best_estimator_

    result = {
        "Country": country,
        "Model": "CART",
        "Best_CV_ROC_AUC": search.best_score_,
        "Best_Parameters": search.best_params_
    }

    cart_results.append(result)

    print("Best parameters:")
    print(search.best_params_)

    print(
        f"Best mean CV ROC-AUC: "
        f"{search.best_score_:.4f}"
    )


                                                                      
CART — Kenya
                                                                      
Best parameters:
{'model__ccp_alpha': 0.0, 'model__max_depth': 3, 'model__min_samples_leaf': 20}
Best mean CV ROC-AUC: 0.7590

                                                                      
CART — Tanzania
                                                                      
Best parameters:
{'model__ccp_alpha': 0.001, 'model__max_depth': 3, 'model__min_samples_leaf': 20}
Best mean CV ROC-AUC: 0.7327

                                                                      
CART — Uganda
                                                                      
Best parameters:
{'model__ccp_alpha': 0.0, 'model__max_depth': 5, 'model__min_samples_leaf': 20}
Best mean CV ROC-AUC: 0.8393


# 10. Model 4 - Random Forest

Random Forest is used as an ensemble tree-based model.

The method combines multiple classification trees constructed from bootstrap samples
and randomized subsets of predictors.

As with CART, missing predictor values are handled inside the modelling pipeline.

The principal Random Forest hyperparameters are tuned using five-fold stratified
cross-validation.

The optimal specification is selected using mean cross-validated ROC-AUC.

In [30]:
# RANDOM FOREST

rf_results = []
rf_models = {}

for country in countries:

    print("\n" + " " * 70)
    print(f"RANDOM FOREST — {country}")
    print(" " * 70)

    Xtr = X_train[country]
    ytr = y_train[country]

    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ])

    param_grid = {
        "model__n_estimators": [
            300,
            500
        ],
        "model__max_depth": [
            None,
            5,
            10
        ],
        "model__max_features": [
            "sqrt",
            "log2"
        ],
        "model__min_samples_leaf": [
            1,
            5,
            10
        ]
    }

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(Xtr, ytr)

    rf_models[country] = search.best_estimator_

    result = {
        "Country": country,
        "Model": "Random Forest",
        "Best_CV_ROC_AUC": search.best_score_,
        "Best_Parameters": search.best_params_
    }

    rf_results.append(result)

    print("Best parameters:")
    print(search.best_params_)

    print(
        f"Best mean CV ROC-AUC: "
        f"{search.best_score_:.4f}"
    )


                                                                      
RANDOM FOREST — Kenya
                                                                      
Best parameters:
{'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 300}
Best mean CV ROC-AUC: 0.8176

                                                                      
RANDOM FOREST — Tanzania
                                                                      
Best parameters:
{'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 500}
Best mean CV ROC-AUC: 0.7699

                                                                      
RANDOM FOREST — Uganda
                                                                      
Best parameters:
{'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 300}
Best mean CV ROC-AUC: 0.9006



# 11. BART-MIA Setup

Bayesian Additive Regression Trees with Missingness Incorporated in Attributes
(BART-MIA) is trained as the fifth model specified in the thesis.

Unlike the four scikit-learn models, BART-MIA is implemented using the R
`bartMachine` package.

The `bartMachine` implementation provides native Missingness Incorporated in
Attributes (MIA) handling through `use_missing_data = TRUE`. This allows missing
predictor values to be incorporated directly into the tree-splitting process
without conventional imputation.

The BART-MIA branch therefore uses the training data generated in
`03_model_preparation.ipynb` while preserving the available missing predictor
values.

Five-fold stratified cross-validation is used for model-development assessment.
The fold assignments are generated in Python and passed to R so that the BART-MIA
cross-validation uses the same five-fold stratified structure as the other models.
### Common Cross-Validation Folds

The same stratified five-fold assignment used for the scikit-learn models is
generated and stored for the BART-MIA analysis.

This ensures that the BART-MIA cross-validation is based on the same training
observations and class-stratification structure.

Fold assignments are generated exclusively from the training data.





In [48]:

# GENERATE AND SAVE COMMON STRATIFIED FOLDS

fold_dir = TRAINING_DIR / "cv_folds"
fold_dir.mkdir(parents=True, exist_ok=True)

fold_assignments = {}

for country in countries:

    Xtr = X_train[country]
    ytr = y_train[country]

    folds = np.empty(len(ytr), dtype=int)

    splitter = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    for fold_number, (_, validation_index) in enumerate(
        splitter.split(Xtr, ytr),
        start=1
    ):
        folds[validation_index] = fold_number

    fold_df = pd.DataFrame({
        "row_index": np.arange(len(ytr)),
        "fold": folds
    })

    fold_path = fold_dir / f"{country}_folds.csv"

    fold_df.to_csv(
        fold_path,
        index=False
    )

    fold_assignments[country] = folds

    print(
        f"{country}: "
        f"{N_SPLITS} folds saved → {fold_path}"
    )

print("\nCommon CV fold generation: PASSED")

Kenya: 5 folds saved → ../results/training/cv_folds/Kenya_folds.csv
Tanzania: 5 folds saved → ../results/training/cv_folds/Tanzania_folds.csv
Uganda: 5 folds saved → ../results/training/cv_folds/Uganda_folds.csv

Common CV fold generation: PASSED


# 12. BART-MIA Training - Kenya

The Kenya training sample is used to estimate the BART-MIA model.

The model is specified as a binary classification BART model.

Missing predictor values are retained and handled through the native MIA mechanism
using `use_missing_data = TRUE`.

Five-fold stratified cross-validation is performed using the previously generated
fold assignments.

The cross-validated probability predictions are used to calculate ROC-AUC.

After cross-validation, a final BART-MIA model is fitted to the complete Kenya
training sample.

The fitted model is saved as an RDS object for subsequent model evaluation.

In [49]:

# BART-MIA — KENYA




country = "Kenya"

# GET TRAINING DATA


Xtr = X_train[country].copy()
ytr = y_train[country].copy()

print(" " * 70)
print(f"BART-MIA — {country}")
print(" " * 70)

print("Training observations:", len(Xtr))
print("Predictors:", Xtr.shape[1])
print("Class 0:", int((ytr == 0).sum()))
print("Class 1:", int((ytr == 1).sum()))


# CREATE DIRECTORIES


bart_input_dir = TRAINING_DIR / "bart_mia_input"
bart_input_dir.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# FILE PATHS


X_path = (
    bart_input_dir /
    f"{country}_X_train.csv"
)

y_path = (
    bart_input_dir /
    f"{country}_y_train.csv"
)

fold_path = (
    fold_dir /
    f"{country}_folds.csv"
)

model_path = (
    MODEL_OUTPUT_DIR /
    f"BART_MIA_{country}.rds"
)

results_path = (
    TRAINING_DIR /
    f"BART_MIA_{country}_results.csv"
)

R_script_path = (
    TRAINING_DIR /
    f"bart_mia_{country.lower()}.R"
)


# VERIFY FOLD FILE


if not fold_path.exists():

    raise FileNotFoundError(
        f"Fold file not found:\n{fold_path}"
    )


# SAVE TRAINING DATA


Xtr.to_csv(
    X_path,
    index=False
)

pd.DataFrame({
    "exporter": ytr.astype(int)
}).to_csv(
    y_path,
    index=False
)

print("\nTraining files saved:")
print("X:", X_path)
print("y:", y_path)
print("folds:", fold_path)

#

R_script = r"""

# BART-MIA — COUNTRY

# Must be set BEFORE loading bartMachine.

options(
    java.parameters = c(
        "-Xmx20g",
        "--add-modules=jdk.incubator.vector",
        "-XX:+UseZGC"
    )
)


# LOAD REQUIRED PACKAGES


library(bartMachine)
library(pROC)

set.seed(RANDOM_STATE)


# LOAD TRAINING DATA


X <- read.csv(
    "X_PATH"
)

y_df <- read.csv(
    "Y_PATH"
)

fold_df <- read.csv(
    "FOLD_PATH"
)


# TARGET


y <- factor(
    y_df$exporter,
    levels = c(0, 1),
    labels = c("0", "1")
)


# CROSS-VALIDATION FOLDS


folds_vec <- fold_df$fold


# BASIC VALIDATION

cat("\n")
cat(" \n")
cat("BART-MIA — COUNTRY\n")
cat(" \n")

cat(
    "Training observations:",
    nrow(X),
    "\n"
)

cat(
    "Predictors:",
    ncol(X),
    "\n"
)

cat(
    "Missing predictor values:",
    sum(is.na(X)),
    "\n"
)

cat(
    "Class 0:",
    sum(y == "0"),
    "\n"
)

cat(
    "Class 1:",
    sum(y == "1"),
    "\n"
)

# VERIFY FOLD STRUCTURE


cat("\nFold distribution:\n")

print(
    table(folds_vec)
)

if (length(folds_vec) != nrow(X)) {

    stop(
        paste0(
            "Fold length mismatch: ",
            "folds = ",
            length(folds_vec),
            ", observations = ",
            nrow(X)
        )
    )
}


# BART-MIA CROSS-VALIDATION


cat("\n")
cat(" \n")
cat("STARTING BART-MIA CROSS-VALIDATION\n")
cat(" \n")

cv_result <- k_fold_cv(
    X = X,
    y = y,
    folds_vec = folds_vec,
    verbose = TRUE,
    num_trees = 50,
    num_burn_in = 250,
    num_iterations_after_burn_in = 1000,
    use_missing_data = TRUE,
    use_missing_data_dummies_as_covars = FALSE,
    seed = RANDOM_STATE
)


# CROSS-VALIDATED PREDICTIONS


cat("\n")
cat(" \n")
cat("CV PREDICTION DIAGNOSTICS\n")
cat(" \n")

cat(
    "Length of y:",
    length(y),
    "\n"
)

# IMPORTANT:
# bartMachine 1.4.2 returns `phat`, not `p_hat`.

cv_prob <- cv_result$phat

cat(
    "Length of phat:",
    length(cv_prob),
    "\n"
)


# CONVERT PREDICTIONS TO NUMERIC


cv_prob <- as.numeric(
    cv_prob
)

y_numeric <- as.numeric(y) - 1

cat(
    "Length of y_numeric:",
    length(y_numeric),
    "\n"
)

cat(
    "Length of cv_prob:",
    length(cv_prob),
    "\n"
)

cat(
    "Missing probabilities:",
    sum(is.na(cv_prob)),
    "\n"
)


# CHECK PREDICTION LENGTH


if (
    length(y_numeric) != length(cv_prob)
) {

    stop(
        paste0(
            "BART-MIA prediction length mismatch: ",
            "y = ",
            length(y_numeric),
            ", cv_prob = ",
            length(cv_prob)
        )
    )
}

# CHECK PROBABILITY RANGE


cat(
    "Minimum predicted probability:",
    min(cv_prob, na.rm = TRUE),
    "\n"
)

cat(
    "Maximum predicted probability:",
    max(cv_prob, na.rm = TRUE),
    "\n"
)


# VALID OBSERVATIONS FOR ROC-AUC


valid <- (
    is.finite(y_numeric) &
    is.finite(cv_prob)
)

cat(
    "Valid ROC-AUC observations:",
    sum(valid),
    "\n"
)

if (
    sum(valid) < 2
) {

    stop(
        "Insufficient valid observations for ROC-AUC."
    )
}


# ROC-AUC


cat("\n")
cat("Calculating ROC-AUC...\n")

roc_result <- pROC::roc(
    response = y_numeric[valid],
    predictor = cv_prob[valid],
    quiet = TRUE,
    direction = "<"
)

cv_auc <- as.numeric(
    pROC::auc(
        roc_result
    )
)

cat(
    "\nBART-MIA CV ROC-AUC:",
    round(cv_auc, 6),
    "\n"
)


# FINAL BART-MIA MODEL


cat("\n")
cat(" \n")
cat("TRAINING FINAL BART-MIA MODEL\n")
cat(" \n")

bart_model <- bartMachine(
    X = X,
    y = y,
    num_trees = 50,
    num_burn_in = 250,
    num_iterations_after_burn_in = 1000,
    use_missing_data = TRUE,
    use_missing_data_dummies_as_covars = FALSE,
    seed = RANDOM_STATE,
    verbose = TRUE
)


# SAVE FINAL MODEL


saveRDS(
    bart_model,
    "MODEL_PATH"
)

cat(
    "\nModel saved to:\nMODEL_PATH\n"
)


# SAVE RESULTS


results <- data.frame(
    Country = "COUNTRY",
    Model = "BART-MIA",
    Best_CV_ROC_AUC = cv_auc
)

write.csv(
    results,
    "RESULTS_PATH",
    row.names = FALSE
)

cat(
    "\nResults saved to:\nRESULTS_PATH\n"
)


# COMPLETION


cat("\n")
cat(" \n")
cat("BART-MIA COUNTRY TRAINING: COMPLETED\n")
cat(" \n")
"""


# INSERT PYTHON VALUES INTO R SCRIPT


R_script = (
    R_script
    .replace(
        "COUNTRY",
        country
    )
    .replace(
        "RANDOM_STATE",
        str(RANDOM_STATE)
    )
    .replace(
        "X_PATH",
        str(X_path)
    )
    .replace(
        "Y_PATH",
        str(y_path)
    )
    .replace(
        "FOLD_PATH",
        str(fold_path)
    )
    .replace(
        "MODEL_PATH",
        str(model_path)
    )
    .replace(
        "RESULTS_PATH",
        str(results_path)
    )
)


# WRITE R SCRIPT


R_script_path.write_text(
    R_script,
    encoding="utf-8"
)

print(
    "\nR script created:"
)

print(
    R_script_path
)


# RUN R SCRIPT


result = subprocess.run(
    [
        R_EXECUTABLE,
        str(R_script_path)
    ],
    capture_output=True,
    text=True
)


# DISPLAY R OUTPUT


print(
    result.stdout
)

if result.stderr:

    print(
        "\nR messages/warnings:"
    )

    print(
        result.stderr
    )


# CHECK EXECUTION STATUS


if result.returncode != 0:

    raise RuntimeError(
        f"BART-MIA {country} training failed."
    )


# VERIFY OUTPUT FILES

print(
    "\nChecking output files..."
)

if not model_path.exists():

    raise FileNotFoundError(
        f"BART-MIA model was not created:\n{model_path}"
    )

if not results_path.exists():

    raise FileNotFoundError(
        f"BART-MIA results file was not created:\n{results_path}"
    )

print(
    f"Model saved: {model_path}"
)

print(
    f" Results saved: {results_path}"
)

print(
    f"\n BART-MIA {country} training completed successfully."
)

                                                                      
BART-MIA — Kenya
                                                                      
Training observations: 819
Predictors: 12
Class 0: 676
Class 1: 143

Training files saved:
X: ../results/training/bart_mia_input/Kenya_X_train.csv
y: ../results/training/bart_mia_input/Kenya_y_train.csv
folds: ../results/training/cv_folds/Kenya_folds.csv

R script created:
../results/training/bart_mia_kenya.R

 
BART-MIA — Kenya
 
Training observations: 819 
Predictors: 12 
Missing predictor values: 0 
Class 0: 676 
Class 1: 143 

Fold distribution:
folds_vec
  1   2   3   4   5 
164 164 164 164 163 

 
STARTING BART-MIA CROSS-VALIDATION
 
.bartMachine initializing with 50 trees...
bartMachine vars checked...
bartMachine java init...
bartMachine factors created...
bartMachine before preprocess...
bartMachine after preprocess... 12 total features...
bartMachine training data finalized...
Now building bartMachine for classification

# 13. BART-MIA Training — Tanzania

The Tanzania training sample is used to estimate the BART-MIA model.

The native MIA mechanism is retained because Tanzania contains substantially more
missing predictor information than Kenya.

The same five-fold stratified cross-validation structure is used.

The held-out Tanzania test observations are not used during this stage.

In [51]:
# ============================================================
# BART-MIA — TANZANIA
# ============================================================

import subprocess
import pandas as pd

country = "Tanzania"

# ============================================================
# GET TRAINING DATA
# ============================================================

Xtr = X_train[country].copy()
ytr = y_train[country].copy()

print("=" * 70)
print(f"BART-MIA — {country}")
print("=" * 70)

print("Training observations:", len(Xtr))
print("Predictors:", Xtr.shape[1])
print("Class 0:", int((ytr == 0).sum()))
print("Class 1:", int((ytr == 1).sum()))

# ============================================================
# CREATE DIRECTORIES
# ============================================================

bart_input_dir.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# FILE PATHS
# ============================================================

X_path = (
    bart_input_dir /
    f"{country}_X_train.csv"
)

y_path = (
    bart_input_dir /
    f"{country}_y_train.csv"
)

fold_path = (
    fold_dir /
    f"{country}_folds.csv"
)

model_path = (
    MODEL_OUTPUT_DIR /
    f"BART_MIA_{country}.rds"
)

results_path = (
    TRAINING_DIR /
    f"BART_MIA_{country}_results.csv"
)

R_script_path = (
    TRAINING_DIR /
    f"bart_mia_{country.lower()}.R"
)

# ============================================================
# VERIFY FOLD FILE
# ============================================================

if not fold_path.exists():

    raise FileNotFoundError(
        f"Fold file not found:\n{fold_path}"
    )

# ============================================================
# SAVE TRAINING DATA
# ============================================================

Xtr.to_csv(
    X_path,
    index=False
)

pd.DataFrame({
    "exporter": ytr.astype(int)
}).to_csv(
    y_path,
    index=False
)

print("\nTraining files saved:")
print("X:", X_path)
print("y:", y_path)
print("folds:", fold_path)

# ============================================================
# R SCRIPT
# ============================================================
#
# IMPORTANT:
# Do NOT use an f-string here.
# This prevents R { } blocks from being interpreted by Python.
# ============================================================

R_script = r"""
# ============================================================
# BART-MIA — COUNTRY
# ============================================================

# ============================================================
# JAVA MEMORY SETTINGS
# ============================================================

options(
    java.parameters = c(
        "-Xmx20g",
        "--add-modules=jdk.incubator.vector",
        "-XX:+UseZGC"
    )
)

# ============================================================
# LOAD REQUIRED PACKAGES
# ============================================================

library(bartMachine)
library(pROC)

set.seed(RANDOM_STATE)

# ============================================================
# LOAD TRAINING DATA
# ============================================================

X <- read.csv(
    "X_PATH"
)

y_df <- read.csv(
    "Y_PATH"
)

fold_df <- read.csv(
    "FOLD_PATH"
)

# ============================================================
# TARGET
# ============================================================

y <- factor(
    y_df$exporter,
    levels = c(0, 1),
    labels = c("0", "1")
)

# ============================================================
# CROSS-VALIDATION FOLDS
# ============================================================

folds_vec <- fold_df$fold

# ============================================================
# BASIC VALIDATION
# ============================================================

cat("\n")
cat("========================================\n")
cat("BART-MIA — COUNTRY\n")
cat("========================================\n")

cat(
    "Training observations:",
    nrow(X),
    "\n"
)

cat(
    "Predictors:",
    ncol(X),
    "\n"
)

cat(
    "Missing predictor values:",
    sum(is.na(X)),
    "\n"
)

cat(
    "Class 0:",
    sum(y == "0"),
    "\n"
)

cat(
    "Class 1:",
    sum(y == "1"),
    "\n"
)

# ============================================================
# VERIFY FOLD STRUCTURE
# ============================================================

cat("\nFold distribution:\n")

print(
    table(folds_vec)
)

if (
    length(folds_vec) != nrow(X)
) {

    stop(
        paste0(
            "Fold length mismatch: ",
            "folds = ",
            length(folds_vec),
            ", observations = ",
            nrow(X)
        )
    )
}

# ============================================================
# BART-MIA CROSS-VALIDATION
# ============================================================

cat("\n")
cat("========================================\n")
cat("STARTING BART-MIA CROSS-VALIDATION\n")
cat("========================================\n")

cv_result <- k_fold_cv(
    X = X,
    y = y,
    folds_vec = folds_vec,
    verbose = TRUE,
    num_trees = 50,
    num_burn_in = 250,
    num_iterations_after_burn_in = 1000,
    use_missing_data = TRUE,
    use_missing_data_dummies_as_covars = FALSE,
    seed = RANDOM_STATE
)

# ============================================================
# CROSS-VALIDATED PREDICTIONS
# ============================================================

cat("\n")
cat("========================================\n")
cat("CV PREDICTION DIAGNOSTICS\n")
cat("========================================\n")

cat(
    "Length of y:",
    length(y),
    "\n"
)

# IMPORTANT:
# bartMachine 1.4.2 returns `phat`, NOT `p_hat`.

cv_prob <- cv_result$phat

cat(
    "Length of phat:",
    length(cv_prob),
    "\n"
)

# ============================================================
# CONVERT PREDICTIONS TO NUMERIC
# ============================================================

cv_prob <- as.numeric(
    cv_prob
)

y_numeric <- as.numeric(y) - 1

cat(
    "Length of y_numeric:",
    length(y_numeric),
    "\n"
)

cat(
    "Length of cv_prob:",
    length(cv_prob),
    "\n"
)

cat(
    "Missing probabilities:",
    sum(is.na(cv_prob)),
    "\n"
)

# ============================================================
# CHECK PREDICTION LENGTH
# ============================================================

if (
    length(y_numeric) != length(cv_prob)
) {

    stop(
        paste0(
            "BART-MIA prediction length mismatch: ",
            "y = ",
            length(y_numeric),
            ", cv_prob = ",
            length(cv_prob)
        )
    )
}

# ============================================================
# CHECK PROBABILITY RANGE
# ============================================================

cat(
    "Minimum predicted probability:",
    min(cv_prob, na.rm = TRUE),
    "\n"
)

cat(
    "Maximum predicted probability:",
    max(cv_prob, na.rm = TRUE),
    "\n"
)

# ============================================================
# VALID OBSERVATIONS FOR ROC-AUC
# ============================================================

valid <- (
    is.finite(y_numeric) &
    is.finite(cv_prob)
)

cat(
    "Valid ROC-AUC observations:",
    sum(valid),
    "\n"
)

if (
    sum(valid) < 2
) {

    stop(
        "Insufficient valid observations for ROC-AUC."
    )
}

# ============================================================
# ROC-AUC
# ============================================================

cat("\nCalculating ROC-AUC...\n")

roc_result <- pROC::roc(
    response = y_numeric[valid],
    predictor = cv_prob[valid],
    quiet = TRUE,
    direction = "<"
)

cv_auc <- as.numeric(
    pROC::auc(
        roc_result
    )
)

cat(
    "\nBART-MIA CV ROC-AUC:",
    round(cv_auc, 6),
    "\n"
)

# ============================================================
# FINAL BART-MIA MODEL
# ============================================================

cat("\n")
cat("========================================\n")
cat("TRAINING FINAL BART-MIA MODEL\n")
cat("========================================\n")

bart_model <- bartMachine(
    X = X,
    y = y,
    num_trees = 50,
    num_burn_in = 250,
    num_iterations_after_burn_in = 1000,
    use_missing_data = TRUE,
    use_missing_data_dummies_as_covars = FALSE,
    seed = RANDOM_STATE,
    verbose = TRUE
)

# ============================================================
# SAVE FINAL MODEL
# ============================================================

saveRDS(
    bart_model,
    "MODEL_PATH"
)

cat(
    "\nModel saved to:\nMODEL_PATH\n"
)

# ============================================================
# SAVE RESULTS
# ============================================================

results <- data.frame(
    Country = "COUNTRY",
    Model = "BART-MIA",
    Best_CV_ROC_AUC = cv_auc
)

write.csv(
    results,
    "RESULTS_PATH",
    row.names = FALSE
)

cat(
    "\nResults saved to:\nRESULTS_PATH\n"
)

# ============================================================
# COMPLETION
# ============================================================

cat("\n")
cat("========================================\n")
cat("BART-MIA COUNTRY TRAINING: COMPLETED\n")
cat("========================================\n")
"""

# ============================================================
# INSERT PYTHON VALUES INTO R SCRIPT
# ============================================================

R_script = (
    R_script
    .replace(
        "COUNTRY",
        country
    )
    .replace(
        "RANDOM_STATE",
        str(RANDOM_STATE)
    )
    .replace(
        "X_PATH",
        str(X_path)
    )
    .replace(
        "Y_PATH",
        str(y_path)
    )
    .replace(
        "FOLD_PATH",
        str(fold_path)
    )
    .replace(
        "MODEL_PATH",
        str(model_path)
    )
    .replace(
        "RESULTS_PATH",
        str(results_path)
    )
)

# ============================================================
# WRITE R SCRIPT
# ============================================================

R_script_path.write_text(
    R_script,
    encoding="utf-8"
)

print(
    "\nR script created:"
)

print(
    R_script_path
)

# ============================================================
# RUN R SCRIPT
# ============================================================

result = subprocess.run(
    [
        R_EXECUTABLE,
        str(R_script_path)
    ],
    capture_output=True,
    text=True
)

# ============================================================
# DISPLAY R OUTPUT
# ============================================================

print(
    result.stdout
)

if result.stderr:

    print(
        "\nR messages/warnings:"
    )

    print(
        result.stderr
    )

# ============================================================
# CHECK EXECUTION STATUS
# ============================================================

if result.returncode != 0:

    raise RuntimeError(
        f"BART-MIA {country} training failed."
    )

# ============================================================
# VERIFY OUTPUT FILES
# ============================================================

print(
    "\nChecking output files..."
)

if not model_path.exists():

    raise FileNotFoundError(
        f"BART-MIA model was not created:\n{model_path}"
    )

if not results_path.exists():

    raise FileNotFoundError(
        f"BART-MIA results file was not created:\n{results_path}"
    )

print(
    f"✓ Model saved: {model_path}"
)

print(
    f"✓ Results saved: {results_path}"
)

print(
    f"\n✓ BART-MIA {country} training completed successfully."
)

BART-MIA — Tanzania
Training observations: 472
Predictors: 12
Class 0: 403
Class 1: 69

Training files saved:
X: ../results/training/bart_mia_input/Tanzania_X_train.csv
y: ../results/training/bart_mia_input/Tanzania_y_train.csv
folds: ../results/training/cv_folds/Tanzania_folds.csv

R script created:
../results/training/bart_mia_tanzania.R

BART-MIA — Tanzania
Training observations: 472 
Predictors: 12 
Missing predictor values: 0 
Class 0: 403 
Class 1: 69 

Fold distribution:
folds_vec
 1  2  3  4  5 
95 95 94 94 94 

STARTING BART-MIA CROSS-VALIDATION
.bartMachine initializing with 50 trees...
bartMachine vars checked...
bartMachine java init...
bartMachine factors created...
bartMachine before preprocess...
bartMachine after preprocess... 12 total features...
bartMachine training data finalized...
Now building bartMachine for classification where "1" is considered the target level...Missing data feature ON. 
building BART with mem-cache speedup...
Iteration 100/1250
Iteration 200/1

# 14. BART-MIA Training — Uganda

The Uganda training sample is used to estimate the BART-MIA model.

Missing predictor observations are retained and incorporated through the MIA
splitting mechanism.

Five-fold stratified cross-validation is used for model-development assessment.

The final Uganda BART-MIA model is subsequently fitted using the complete Uganda
training sample and saved for evaluation in `05_model_evaluation.ipynb`.

In [52]:
# ============================================================
# BART-MIA — UGANDA
# ============================================================

import subprocess
import pandas as pd

country = "Uganda"

# ============================================================
# GET TRAINING DATA
# ============================================================

Xtr = X_train[country].copy()
ytr = y_train[country].copy()

print("=" * 70)
print(f"BART-MIA — {country}")
print("=" * 70)

print("Training observations:", len(Xtr))
print("Predictors:", Xtr.shape[1])
print("Class 0:", int((ytr == 0).sum()))
print("Class 1:", int((ytr == 1).sum()))

# ============================================================
# CREATE DIRECTORIES
# ============================================================

bart_input_dir.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# FILE PATHS
# ============================================================

X_path = (
    bart_input_dir /
    f"{country}_X_train.csv"
)

y_path = (
    bart_input_dir /
    f"{country}_y_train.csv"
)

fold_path = (
    fold_dir /
    f"{country}_folds.csv"
)

model_path = (
    MODEL_OUTPUT_DIR /
    f"BART_MIA_{country}.rds"
)

results_path = (
    TRAINING_DIR /
    f"BART_MIA_{country}_results.csv"
)

R_script_path = (
    TRAINING_DIR /
    f"bart_mia_{country.lower()}.R"
)

# ============================================================
# VERIFY FOLD FILE
# ============================================================

if not fold_path.exists():

    raise FileNotFoundError(
        f"Fold file not found:\n{fold_path}"
    )

# ============================================================
# SAVE TRAINING DATA
# ============================================================

Xtr.to_csv(
    X_path,
    index=False
)

pd.DataFrame({
    "exporter": ytr.astype(int)
}).to_csv(
    y_path,
    index=False
)

print("\nTraining files saved:")
print("X:", X_path)
print("y:", y_path)
print("folds:", fold_path)

# ============================================================
# R SCRIPT
# ============================================================
#
# IMPORTANT:
# Do NOT use an f-string here.
# This prevents R { } blocks from being interpreted by Python.
# ============================================================

R_script = r"""
# ============================================================
# BART-MIA — COUNTRY
# ============================================================

# ============================================================
# JAVA MEMORY SETTINGS
# ============================================================

options(
    java.parameters = c(
        "-Xmx20g",
        "--add-modules=jdk.incubator.vector",
        "-XX:+UseZGC"
    )
)

# ============================================================
# LOAD REQUIRED PACKAGES
# ============================================================

library(bartMachine)
library(pROC)

set.seed(RANDOM_STATE)

# ============================================================
# LOAD TRAINING DATA
# ============================================================

X <- read.csv(
    "X_PATH"
)

y_df <- read.csv(
    "Y_PATH"
)

fold_df <- read.csv(
    "FOLD_PATH"
)

# ============================================================
# TARGET
# ============================================================

y <- factor(
    y_df$exporter,
    levels = c(0, 1),
    labels = c("0", "1")
)

# ============================================================
# CROSS-VALIDATION FOLDS
# ============================================================

folds_vec <- fold_df$fold

# ============================================================
# BASIC VALIDATION
# ============================================================

cat("\n")
cat("========================================\n")
cat("BART-MIA — COUNTRY\n")
cat("========================================\n")

cat(
    "Training observations:",
    nrow(X),
    "\n"
)

cat(
    "Predictors:",
    ncol(X),
    "\n"
)

cat(
    "Missing predictor values:",
    sum(is.na(X)),
    "\n"
)

cat(
    "Class 0:",
    sum(y == "0"),
    "\n"
)

cat(
    "Class 1:",
    sum(y == "1"),
    "\n"
)

# ============================================================
# VERIFY FOLD STRUCTURE
# ============================================================

cat("\nFold distribution:\n")

print(
    table(folds_vec)
)

if (
    length(folds_vec) != nrow(X)
) {

    stop(
        paste0(
            "Fold length mismatch: ",
            "folds = ",
            length(folds_vec),
            ", observations = ",
            nrow(X)
        )
    )
}

# ============================================================
# BART-MIA CROSS-VALIDATION
# ============================================================

cat("\n")
cat("========================================\n")
cat("STARTING BART-MIA CROSS-VALIDATION\n")
cat("========================================\n")

cv_result <- k_fold_cv(
    X = X,
    y = y,
    folds_vec = folds_vec,
    verbose = TRUE,
    num_trees = 50,
    num_burn_in = 250,
    num_iterations_after_burn_in = 1000,
    use_missing_data = TRUE,
    use_missing_data_dummies_as_covars = FALSE,
    seed = RANDOM_STATE
)

# ============================================================
# CROSS-VALIDATED PREDICTIONS
# ============================================================

cat("\n")
cat("========================================\n")
cat("CV PREDICTION DIAGNOSTICS\n")
cat("========================================\n")

cat(
    "Length of y:",
    length(y),
    "\n"
)

# IMPORTANT:
# bartMachine 1.4.2 returns `phat`, NOT `p_hat`.

cv_prob <- cv_result$phat

cat(
    "Length of phat:",
    length(cv_prob),
    "\n"
)

# ============================================================
# CONVERT PREDICTIONS TO NUMERIC
# ============================================================

cv_prob <- as.numeric(
    cv_prob
)

y_numeric <- as.numeric(y) - 1

cat(
    "Length of y_numeric:",
    length(y_numeric),
    "\n"
)

cat(
    "Length of cv_prob:",
    length(cv_prob),
    "\n"
)

cat(
    "Missing probabilities:",
    sum(is.na(cv_prob)),
    "\n"
)

# ============================================================
# CHECK PREDICTION LENGTH
# ============================================================

if (
    length(y_numeric) != length(cv_prob)
) {

    stop(
        paste0(
            "BART-MIA prediction length mismatch: ",
            "y = ",
            length(y_numeric),
            ", cv_prob = ",
            length(cv_prob)
        )
    )
}

# ============================================================
# CHECK PROBABILITY RANGE
# ============================================================

cat(
    "Minimum predicted probability:",
    min(cv_prob, na.rm = TRUE),
    "\n"
)

cat(
    "Maximum predicted probability:",
    max(cv_prob, na.rm = TRUE),
    "\n"
)

# ============================================================
# VALID OBSERVATIONS FOR ROC-AUC
# ============================================================

valid <- (
    is.finite(y_numeric) &
    is.finite(cv_prob)
)

cat(
    "Valid ROC-AUC observations:",
    sum(valid),
    "\n"
)

if (
    sum(valid) < 2
) {

    stop(
        "Insufficient valid observations for ROC-AUC."
    )
}

# ============================================================
# ROC-AUC
# ============================================================

cat("\nCalculating ROC-AUC...\n")

roc_result <- pROC::roc(
    response = y_numeric[valid],
    predictor = cv_prob[valid],
    quiet = TRUE,
    direction = "<"
)

cv_auc <- as.numeric(
    pROC::auc(
        roc_result
    )
)

cat(
    "\nBART-MIA CV ROC-AUC:",
    round(cv_auc, 6),
    "\n"
)

# ============================================================
# FINAL BART-MIA MODEL
# ============================================================

cat("\n")
cat("========================================\n")
cat("TRAINING FINAL BART-MIA MODEL\n")
cat("========================================\n")

bart_model <- bartMachine(
    X = X,
    y = y,
    num_trees = 50,
    num_burn_in = 250,
    num_iterations_after_burn_in = 1000,
    use_missing_data = TRUE,
    use_missing_data_dummies_as_covars = FALSE,
    seed = RANDOM_STATE,
    verbose = TRUE
)

# ============================================================
# SAVE FINAL MODEL
# ============================================================

saveRDS(
    bart_model,
    "MODEL_PATH"
)

cat(
    "\nModel saved to:\nMODEL_PATH\n"
)

# ============================================================
# SAVE RESULTS
# ============================================================

results <- data.frame(
    Country = "COUNTRY",
    Model = "BART-MIA",
    Best_CV_ROC_AUC = cv_auc
)

write.csv(
    results,
    "RESULTS_PATH",
    row.names = FALSE
)

cat(
    "\nResults saved to:\nRESULTS_PATH\n"
)

# ============================================================
# COMPLETION
# ============================================================

cat("\n")
cat("========================================\n")
cat("BART-MIA COUNTRY TRAINING: COMPLETED\n")
cat("========================================\n")
"""

# ============================================================
# INSERT PYTHON VALUES INTO R SCRIPT
# ============================================================

R_script = (
    R_script
    .replace(
        "COUNTRY",
        country
    )
    .replace(
        "RANDOM_STATE",
        str(RANDOM_STATE)
    )
    .replace(
        "X_PATH",
        str(X_path)
    )
    .replace(
        "Y_PATH",
        str(y_path)
    )
    .replace(
        "FOLD_PATH",
        str(fold_path)
    )
    .replace(
        "MODEL_PATH",
        str(model_path)
    )
    .replace(
        "RESULTS_PATH",
        str(results_path)
    )
)

# ============================================================
# WRITE R SCRIPT
# ============================================================

R_script_path.write_text(
    R_script,
    encoding="utf-8"
)

print(
    "\nR script created:"
)

print(
    R_script_path
)

# ============================================================
# RUN R SCRIPT
# ============================================================

result = subprocess.run(
    [
        R_EXECUTABLE,
        str(R_script_path)
    ],
    capture_output=True,
    text=True
)

# ============================================================
# DISPLAY R OUTPUT
# ============================================================

print(
    result.stdout
)

if result.stderr:

    print(
        "\nR messages/warnings:"
    )

    print(
        result.stderr
    )

# ============================================================
# CHECK EXECUTION STATUS
# ============================================================

if result.returncode != 0:

    raise RuntimeError(
        f"BART-MIA {country} training failed."
    )

# ============================================================
# VERIFY OUTPUT FILES
# ============================================================

print(
    "\nChecking output files..."
)

if not model_path.exists():

    raise FileNotFoundError(
        f"BART-MIA model was not created:\n{model_path}"
    )

if not results_path.exists():

    raise FileNotFoundError(
        f"BART-MIA results file was not created:\n{results_path}"
    )

print(
    f"✓ Model saved: {model_path}"
)

print(
    f"✓ Results saved: {results_path}"
)

print(
    f"\n✓ BART-MIA {country} training completed successfully."
)

BART-MIA — Uganda
Training observations: 484
Predictors: 12
Class 0: 431
Class 1: 53

Training files saved:
X: ../results/training/bart_mia_input/Uganda_X_train.csv
y: ../results/training/bart_mia_input/Uganda_y_train.csv
folds: ../results/training/cv_folds/Uganda_folds.csv

R script created:
../results/training/bart_mia_uganda.R

BART-MIA — Uganda
Training observations: 484 
Predictors: 12 
Missing predictor values: 0 
Class 0: 431 
Class 1: 53 

Fold distribution:
folds_vec
 1  2  3  4  5 
97 97 97 97 96 

STARTING BART-MIA CROSS-VALIDATION
.bartMachine initializing with 50 trees...
bartMachine vars checked...
bartMachine java init...
bartMachine factors created...
bartMachine before preprocess...
bartMachine after preprocess... 12 total features...
bartMachine training data finalized...
Now building bartMachine for classification where "1" is considered the target level...Missing data feature ON. 
building BART with mem-cache speedup...
Iteration 100/1250
Iteration 200/1250
Iteratio

# 15. BART-MIA Cross-Validation Results

The out-of-fold BART-MIA probability predictions are used to calculate the
cross-validated ROC-AUC for each country.

These results represent model-development performance and are therefore
comparable with the cross-validated ROC-AUC values produced for Logistic
Regression, LASSO Logistic Regression, CART, and Random Forest.

No held-out test observations are used here.

The resulting BART-MIA models and cross-validation results are retained for the
combined model comparison.

In [54]:

# COLLECT BART-MIA RESULTS


bart_results = []

for country in countries:

    result_path = (
        TRAINING_DIR /
        f"BART_MIA_{country}_results.csv"
    )

    if not result_path.exists():
        raise FileNotFoundError(
            f"BART-MIA result not found for {country}: "
            f"{result_path}"
        )

    result_df = pd.read_csv(result_path)

    bart_results.append(result_df)

bart_results_df = pd.concat(
    bart_results,
    ignore_index=True
)

print("BART-MIA cross-validation results")
print(" " * 70)

display(bart_results_df)

print("\nBART-MIA result collection: PASSED")

BART-MIA cross-validation results
                                                                      


,Country,Model,Best_CV_ROC_AUC
0,Kenya,BART-MIA,0.818503
1,Tanzania,BART-MIA,0.715359
2,Uganda,BART-MIA,0.885041



BART-MIA result collection: PASSED


## 8. Define Hyperparameter Search Spaces

Hyperparameter grids are defined for each candidate model.

The hyperparameters are selected to control model complexity and regularise
the models where appropriate.

Hyperparameter selection is performed using cross-validation on the training
data only.

In [ ]:

# HYPERPARAMETER GRIDS


param_grids = {

    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100]
    },

    "LASSO Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__l1_ratio": [1.0]
    },

    "CART": {
        "model__max_depth": [3, 5, 7, 10, None],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 5]
    },

    "Random Forest": {
        "model__n_estimators": [200, 500],
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 5],
        "model__max_features": ["sqrt", "log2"]
    }
}

for model_name, grid in param_grids.items():

    print("\n" + model_name)
    print(" " * 50)

    for parameter, values in grid.items():
        print(parameter, ":", values)


Logistic Regression
                                                  
model__C : [0.01, 0.1, 1, 10, 100]

LASSO Logistic Regression
                                                  
model__C : [0.01, 0.1, 1, 10, 100]
model__l1_ratio : [1.0]

CART
                                                  
model__max_depth : [3, 5, 7, 10, None]
model__min_samples_split : [2, 5, 10]
model__min_samples_leaf : [1, 2, 5]

Random Forest
                                                  
model__n_estimators : [200, 500]
model__max_depth : [None, 5, 10, 20]
model__min_samples_split : [2, 5, 10]
model__min_samples_leaf : [1, 2, 5]
model__max_features : ['sqrt', 'log2']


## 9. Train and Tune Candidate Models

Each candidate model is tuned separately for each country.

GridSearchCV evaluates the specified hyperparameter combinations using
five-fold stratified cross-validation.

ROC-AUC is maximised during model selection.

The independent test data are not used in this process.

In [14]:
# MODEL TRAINING AND HYPERPARAMETER TUNING
training_results = []

best_estimators = {}

best_parameters = {}

best_cv_scores = {}


for country in countries:

    print("\n" + " " * 80)
    print(f"COUNTRY: {country}")
    print(" " * 80)

    X_train = X_train_smote[country]
    y_train = y_train_smote[country]

    best_estimators[country] = {}
    best_parameters[country] = {}
    best_cv_scores[country] = {}

    for model_name, pipeline in models.items():

        print("\n" + " " * 70)
        print(
            f"{model_name} — {country}"
        )
        print(" " * 70)

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grids[model_name],
            scoring=scoring,
            cv=cv,
            n_jobs=-1,
            refit=True,
            return_train_score=False
        )

        grid_search.fit(
            X_train,
            y_train
        )

        best_estimators[country][
            model_name
        ] = grid_search.best_estimator_

        best_parameters[country][
            model_name
        ] = grid_search.best_params_

        best_cv_scores[country][
            model_name
        ] = grid_search.best_score_

        training_results.append({
            "Country": country,
            "Model": model_name,
            "Best_CV_ROC_AUC": grid_search.best_score_,
            "Best_Parameters": grid_search.best_params_
        })

        print(
            "Best parameters:"
        )

        print(
            grid_search.best_params_
        )

        print(
            f"Best mean CV ROC-AUC: "
            f"{grid_search.best_score_:.4f}"
        )


                                                                                
COUNTRY: Kenya
                                                                                

                                                                      
Logistic Regression — Kenya
                                                                      
Best parameters:
{'model__C': 100}
Best mean CV ROC-AUC: 0.8431

                                                                      
LASSO Logistic Regression — Kenya
                                                                      
Best parameters:
{'model__C': 10, 'model__l1_ratio': 1.0}
Best mean CV ROC-AUC: 0.8429

                                                                      
CART — Kenya
                                                                      
Best parameters:
{'model__max_depth': 7, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2}
Best mean CV ROC-AUC: 0.8792

                                   

## 10. Compile Cross-Validation Results

The best cross-validation performance and selected hyperparameters for every
model-country combination are consolidated into a single table.

This table is used to compare the candidate models before selecting the
preferred model for each country.

In [15]:
# COMPILE TRAINING RESULTS
training_results_df = pd.DataFrame(
    training_results
)

training_results_df = (
    training_results_df
    .sort_values(
        [
            "Country",
            "Best_CV_ROC_AUC"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

display(
    training_results_df
)

,Country,Model,Best_CV_ROC_AUC,Best_Parameters
0,Kenya,Random Forest,0.957440,"{'model__max_depth': 20, 'model__max_features'..."
1,Kenya,CART,0.879186,"{'model__max_depth': 7, 'model__min_samples_le..."
2,Kenya,Logistic Regression,0.843071,{'model__C': 100}
3,Kenya,LASSO Logistic Regression,0.842929,"{'model__C': 10, 'model__l1_ratio': 1.0}"
4,Tanzania,Random Forest,0.973293,"{'model__max_depth': None, 'model__max_feature..."
5,Tanzania,CART,0.925295,"{'model__max_depth': 7, 'model__min_samples_le..."
6,Tanzania,LASSO Logistic Regression,0.747088,"{'model__C': 0.1, 'model__l1_ratio': 1.0}"
7,Tanzania,Logistic Regression,0.745958,{'model__C': 100}
8,Uganda,Random Forest,0.990479,"{'model__max_depth': 20, 'model__max_features'..."
9,Uganda,CART,0.941038,"{'model__max_depth': 7, 'model__min_samples_le..."


## 11. Compare Candidate Models

For each country, candidate models are ranked according to their mean
cross-validated ROC-AUC.

The model with the highest cross-validated ROC-AUC is identified as the
preferred model for that country.

This selection is based exclusively on the training data.

In [16]:
# MODEL RANKING BY COUNTRY

model_ranking = (
    training_results_df
    .copy()
)

model_ranking["Rank"] = (
    model_ranking
    .groupby("Country")[
        "Best_CV_ROC_AUC"
    ]
    .rank(
        method="min",
        ascending=False
    )
)

model_ranking = (
    model_ranking
    .sort_values(
        [
            "Country",
            "Rank"
        ]
    )
)

display(
    model_ranking
)

,Country,Model,Best_CV_ROC_AUC,Best_Parameters,Rank
0,Kenya,Random Forest,0.957440,"{'model__max_depth': 20, 'model__max_features'...",1.0
1,Kenya,CART,0.879186,"{'model__max_depth': 7, 'model__min_samples_le...",2.0
2,Kenya,Logistic Regression,0.843071,{'model__C': 100},3.0
3,Kenya,LASSO Logistic Regression,0.842929,"{'model__C': 10, 'model__l1_ratio': 1.0}",4.0
4,Tanzania,Random Forest,0.973293,"{'model__max_depth': None, 'model__max_feature...",1.0
5,Tanzania,CART,0.925295,"{'model__max_depth': 7, 'model__min_samples_le...",2.0
6,Tanzania,LASSO Logistic Regression,0.747088,"{'model__C': 0.1, 'model__l1_ratio': 1.0}",3.0
7,Tanzania,Logistic Regression,0.745958,{'model__C': 100},4.0
8,Uganda,Random Forest,0.990479,"{'model__max_depth': 20, 'model__max_features'...",1.0
9,Uganda,CART,0.941038,"{'model__max_depth': 7, 'model__min_samples_le...",2.0


## 12. Select Preferred Model for Each Country

The highest-performing candidate according to mean cross-validated ROC-AUC is
selected for each country.

The held-out test data are not used to make this selection.

In [17]:
# SELECT BEST MODEL PER COUNTRY

best_model_by_country = {}

for country in countries:

    country_results = training_results_df[
        training_results_df["Country"] == country
    ].copy()

    best_row = (
        country_results
        .sort_values(
            "Best_CV_ROC_AUC",
            ascending=False
        )
        .iloc[0]
    )

    best_model_by_country[country] = {
        "Model": best_row["Model"],
        "CV_ROC_AUC": best_row[
            "Best_CV_ROC_AUC"
        ],
        "Parameters": best_row[
            "Best_Parameters"
        ]
    }


best_model_summary = pd.DataFrame([
    {
        "Country": country,
        "Best_Model": values["Model"],
        "Best_CV_ROC_AUC": values["CV_ROC_AUC"],
        "Best_Parameters": values["Parameters"]
    }

    for country, values
    in best_model_by_country.items()
])


display(
    best_model_summary
)

,Country,Best_Model,Best_CV_ROC_AUC,Best_Parameters
0,Kenya,Random Forest,0.957440,"{'model__max_depth': 20, 'model__max_features'..."
1,Tanzania,Random Forest,0.973293,"{'model__max_depth': None, 'model__max_feature..."
2,Uganda,Random Forest,0.990479,"{'model__max_depth': 20, 'model__max_features'..."


## 13. Expected Model-Selection Output

The preferred model for each country is determined by the highest mean
cross-validated ROC-AUC.

The previously obtained training results provide the following benchmark:

- Kenya → Random Forest
- Tanzania → Random Forest
- Uganda → Logistic Regression

These values are not manually imposed in the code. They should emerge from
the cross-validation procedure using the prepared data.

In [18]:
# MODEL-SELECTION CHECK

for country in countries:

    selected_model = (
        best_model_by_country[country]["Model"]
    )

    selected_score = (
        best_model_by_country[country]["CV_ROC_AUC"]
    )

    print(
        f"{country} → "
        f"{selected_model} | "
        f"CV ROC-AUC = {selected_score:.4f}"
    )

Kenya → Random Forest | CV ROC-AUC = 0.9574
Tanzania → Random Forest | CV ROC-AUC = 0.9733
Uganda → Random Forest | CV ROC-AUC = 0.9905
